In [ ]:
include( "../args.jl" )
include( "../model.jl" )
include( "../cluster.jl" )
include( "../prob.jl" )
include( "../visual.jl" )

include( "../0-params/rho-list.jl" )
include( "../0-params/nos-list.jl" )

using Optim

home, base = expfolder( "system" )

colormap = cgrad( :ice; rev=true )

In [ ]:
# Hyper parameters.
N = 128;  s = 0.01

# Create folder and extract critical noise.
folderlist = [createfolder( findfolder( N, ρ, s; base=base, warn=false, crit=true ) ) for ρ ∈ ρlist]

# Extract critical noise values.
ηclist = first.( readdlm.( folderlist.*"../order-0.5.txt" ) );

In [ ]:
# Initialize progress meter.
meter = Progress( Nρ, "Importing data."; dt=1.0 )

# Initialize data variables.
Fdata = Vector{Matrix{defInt}}( undef, Nρ )
θdata = Vector{Matrix{defFloat}}( undef, Nρ )

# Import cluster dynamical data.
for i ∈ 1:Nρ
    # Check that the folder path is valid.
    check = ispath( folderlist[i]*"frequency-map.txt" )

    # Unpack the frequency data and the optimal rates.
    Fdata[i] = check ? round.( defInt, readdlm( folderlist[i]*"frequency-map.txt" ) ) : nothing
    θdata[i] = readdlm( folderlist[i]*"optimal-rates-hat.txt" )

    # Update progress meter.
    next!( meter )
end

In [ ]:
# Focal parameters to create heatmap.
n = 64       # Subset to consider.
i = Nρ - 3   # Density.

# Extract change frequency and build probability map.
ΔFlist = sum( Fdata[i][1:n,1:n]; dims=2 )
Ω = Fdata[i][1:n,1:n]./ΔFlist

# Extract closest cluster mass distribution.
iη = sortperm( abs.( ηlist .- ηclist[i] ) )[1]
ΔI = round.( defInt, vec( readdlm(
    findfolder( N, ρlist[i], s, ηlist[iη]; base=base )*"cluster-frequency.txt" ) ) )
Δi = ΔI./sum( ΔI )

# Extrat power-law exponent.
λ = 10.0^optimize( createzipfloss( N, ΔI ), [log10( 1.5 )] ).minimizer[1];

In [ ]:
# Subset of cluster population to consider.
x = repeat( 1:n, inner=n )
y = repeat( 1:n, outer=n )
z = vec( Ω[1:n,1:n] )

# Plot a single row of the cluster change distribution.
plt = plot(; layout=@layout( [[a{0.35h}; [b{0.95w} c]] [d; e; f]] ), size=(650,450) )
plot!( plt; right_margin=10pt, legend=false )

# Plot the cluster mass distribution.
ιlist = 0.0 .< Δi
scatter!( plt[1], (1:N)[ιlist], Δi[ιlist]; color=:gray, markerstrokewidth=0 )
plot!( plt[1], 1:N, zipf.( 1:N; b=N, λ=λ ); color=:black, lw=2 )

ymin = floor( log10( minimum( Δi[ιlist] ) ) ) - 1
xticks = round.( defInt, 2.0.^(0:2:log2( N )) )
plot!( plt[1]; xlims=(1,N), xscale=:log10, xticks=(xticks,string.(xticks)) )
plot!( plt[1]; ylims=10.0.^(ymin,0), yscale=:log10 )
plot!( plt[1]; xlabel="cluster size, "*L"i", ylabel="probability, "*L"P(i)" )
plot!( plt[1]; title="(a)", titleposition=:left )

# Plot the probability map.
plot!( plt[2]; right_margin=0pt )

zmax = 0.1ceil( maximum( z )/0.1 )
scatter!( plt[2], x, -y; marker_z=z, color=colormap, colorbar=false,
    clims=(0,zmax), markersize=1.5, edgecolor=:transparent, markerstrokewidth=0 )

plot!( plt[2]; xlims=(0, n+1), xticks=0:n/4:n )
plot!( plt[2]; ylims=(-n-1,0), yticks=(0:-n/4:-n, 0:round.( defInt, n/4 ):n ) )
plot!( plt[2]; xlabel="new cluster size, "*L"j", ylabel="original cluster size, "*L"i" )
plot!( plt[2]; title="(b)", titleposition=:left )

# Plot the colorbar.
m = 1000
vals = reshape( range( 0,0.5,length=m ),:,1 )
plot!( plt[3]; right_margin=15pt )
heatmap!( plt[3], vals; color=colormap, clims=(0,0.5),
    colorbar=false, xtick=false, xticks=false,
    framestyle=:box, mirror=true, yticks=(0:200:1000,0.0:0.1:0.5) )
plot!( plt[3]; ylabel="probability, "*L"ω_{i,j}" )

# Plot the row from Ω associated with i=1.
k = 1;  flist = Ω[k,:]./sum( Ω[k,:] )
ιlist = (1:n .< n) .& (0.0 .< flist)
scatter!( plt[4], (1:n)[ιlist], flist[ιlist]; color=:gray, markerstrokewidth=0 )

θ = θdata[i][k,:]
plot!( plt[4], 1:n, P.( 1:n; N=n, i=k, λℓ=θ[1], λr=θ[2], κ=θ[3] ); color=:black, lw=2 )

xticks = round.( defInt, 2.0.^(0:round( defInt, log2( n ) + 1 )) )
ymin = floor( log10( minimum( flist[ιlist] ) ) )
plot!( plt[4]; xlims=(1,xticks[end]+1), xscale=:log10, xticks=(xticks,string.( xticks )) )
plot!( plt[4]; ylims=10.0.^(ymin,0), yscale=:log10 )
plot!( plt[4]; xlabel="new cluster size, "*L"j", ylabel="probability, "*latexstring( "ω_{$(k),j}" ) )
plot!( plt[4]; title="(c)", titleposition=:left )

# Plot the row from Ω associated with i=n/2.
k = round( defInt, n/2 );  flist = Ω[k,:]
ιlist = (1:n .< n) .& (0.0 .< flist)
scatter!( plt[5], (1:n)[ιlist], flist[ιlist]; color=:gray, markerstrokewidth=0 )

θ = θdata[i][k,:]
plot!( plt[5], 1:n, P.( 1:n; N=n, i=k, λℓ=θ[1], λr=θ[2], κ=θ[3] ); color=:black, lw=2 )

ymin = floor( log10( minimum( flist[ιlist] ) ) )
plot!( plt[5]; xlims=(1,xticks[end]+1), xscale=:log10, xticks=(xticks,string.( xticks )) )
plot!( plt[5]; ylims=10.0.^(ymin,0), yscale=:log10 )
plot!( plt[5]; xlabel="new cluster size, "*L"j", ylabel="probability, "*latexstring( "ω_{$(k),j}" ) )
plot!( plt[5]; title="(d)", titleposition=:left )

# Plot the row from Ω associated with i=n.
k = n;  flist = Ω[k,:]
ιlist = (1:n .< n) .& (0.0 .< flist)
scatter!( plt[6], (1:n)[ιlist], flist[ιlist]; color=:gray, markerstrokewidth=0 )

θ = θdata[i][k,:]
plot!( plt[6], 1:n, P.( 1:n; N=n, i=k, λℓ=θ[1], λr=θ[2], κ=θ[3] ); color=:black, lw=2 )

ymin = floor( log10( minimum( flist[ιlist] ) ) )
plot!( plt[6]; xlims=(1,xticks[end]+1), xscale=:log10, xticks=(xticks,string.( xticks )) )
plot!( plt[6]; ylims=10.0.^(ymin,0), yscale=:log10 )
plot!( plt[6]; xlabel="new cluster size, "*L"j", ylabel="probability, "*latexstring( "ω_{$(k),j}" ) )
plot!( plt[6]; title="(e)", titleposition=:left )

# saveplot( plt, figurefolder*"next-step-distribution_N-$(N)_n-$(n)"; dpi=600 );

In [ ]:
# if true
    # Visualize simulation with a gif.
    # N̂ = n
    # gvar = creategif( N, N̂, f )
    # gif( gvar, "cluster-change-prob_N-$(N).gif" )
# end